# Your first compiled Python loop

**Core notebook, about 25 minutes.** We will time a Python loop, ask Numba to make a faster version, check that the answer is unchanged, and time it again.

The important habit is: **check, time, change, check again, then time again.**

In [ ]:
import numpy as np
import numba
from numba import jit

print("NumPy", np.__version__)
print("Numba", numba.__version__)

## 1. Time the current Python version

This function contains an explicit loop. In spatial analysis, we often filter points by distance from an origin and accumulate values for points within a cutoff radius. It is intentionally written as clear Python before we optimize it.

In [ ]:
def radial_distance_sum_python(x, y, cx, cy, radius_cutoff):
    total = 0.0
    for i in range(x.shape[0]):
        dx = x[i] - cx
        dy = y[i] - cy
        r = (dx * dx + dy * dy) ** 0.5
        if r < radius_cutoff:
            total += r
    return total

rng = np.random.default_rng(2026)
n = 1_000_000
x = rng.uniform(-5, 5, size=n)
y = rng.uniform(-5, 5, size=n)
cx, cy = 0.0, 0.0
radius_cutoff = 3.0

python_result = radial_distance_sum_python(x, y, cx, cy, radius_cutoff)
python_time = %timeit -o -n 10 -r 3 radial_distance_sum_python(x, y, cx, cy, radius_cutoff)

## 2. Add `@jit`, call once, and check the answer

`@jit` asks Numba to compile the function, which means making a machine-code version that can run faster. Numba does this work during the first call, so time later calls instead of the first one.

In [ ]:
@jit
def radial_distance_sum_numba(x, y, cx, cy, radius_cutoff):
    total = 0.0
    for i in range(x.shape[0]):
        dx = x[i] - cx
        dy = y[i] - cy
        r = (dx * dx + dy * dy) ** 0.5
        if r < radius_cutoff:
            total += r
    return total

# First call: make the compiled version and return an answer.
compiled_result = radial_distance_sum_numba(x, y, cx, cy, radius_cutoff)

# Correctness comes before speed.
np.testing.assert_allclose(compiled_result, python_result, rtol=1e-12)

# The compiled version already exists, so this is a fair timing.
numba_time = %timeit -o -n 10 -r 3 radial_distance_sum_numba(x, y, cx, cy, radius_cutoff)
print(f"Speedup after the first call: {python_time.average / numba_time.average:.1f}x")

## 3. Compare with NumPy

Numba is not a replacement for clear NumPy. If NumPy can express the work in one clear line, time both versions. Here NumPy uses `np.hypot` and boolean indexing.

In [ ]:
def radial_distance_sum_numpy(x, y, cx, cy, radius_cutoff):
    r = np.hypot(x - cx, y - cy)
    return np.sum(r[r < radius_cutoff])

numpy_result = radial_distance_sum_numpy(x, y, cx, cy, radius_cutoff)
np.testing.assert_allclose(numpy_result, python_result, rtol=1e-12)
numpy_time = %timeit -o -n 10 -r 3 radial_distance_sum_numpy(x, y, cx, cy, radius_cutoff)

print(f"Python: {python_time.average * 1e3:.3f} ms")
print(f"Numba:  {numba_time.average * 1e3:.3f} ms")
print(f"NumPy:  {numpy_time.average * 1e3:.3f} ms")

## Your turn: 3-point moving average

**12 minutes.** In signal processing and time-series analysis, smoothing data with a moving average is a standard task. Write a function `moving_average` that computes a 3-point centered moving average `out[i] = (data[i-1] + data[i] + data[i+1]) / 3.0` for array indices `1` to `N-2` using a Python `for` loop. Then:

1. Compile it with `@jit`.
2. Check it against the NumPy array slice expression `(data[:-2] + data[1:-1] + data[2:]) / 3.0`.
3. Warm up before timing.
4. Time Numba and NumPy.
5. Explain the result in one sentence.

```python
def moving_average(data):
    n = data.shape[0]
    out = np.empty(n - 2, dtype=data.dtype)
    # Add your loop over indices 1 to n - 1.
    return out
```

Blue sticky note means you need help. Yellow means you are ready to discuss.

In [ ]:
# Write and test your solution here before opening the solution cell below.

<details><summary>Solution and discussion</summary>

Run the solution cell below after attempting the exercise. Both NumPy array slicing and Numba provide fast results. Numba avoids intermediate temporary array allocations during slicing and arithmetic, combining high speed with clear direct Python loop code.

```python
@jit
def moving_average_numba(data):
    n = data.shape[0]
    out = np.empty(n - 2, dtype=data.dtype)
    for i in range(1, n - 1):
        out[i - 1] = (data[i - 1] + data[i] + data[i + 1]) / 3.0
    return out
```

</details>

In [ ]:
@jit
def moving_average_numba(data):
    n = data.shape[0]
    out = np.empty(n - 2, dtype=data.dtype)
    for i in range(1, n - 1):
        out[i - 1] = (data[i - 1] + data[i] + data[i + 1]) / 3.0
    return out

rng = np.random.default_rng(2026)
data = rng.random(1_000_000)

expected = (data[:-2] + data[1:-1] + data[2:]) / 3.0
actual = moving_average_numba(data)  # compile and warm up
np.testing.assert_allclose(actual, expected, rtol=1e-12)

exercise_numba_time = %timeit -o -n 10 -r 3 moving_average_numba(data)
exercise_numpy_time = %timeit -o -n 10 -r 3 (data[:-2] + data[1:-1] + data[2:]) / 3.0
print(f"Numba: {exercise_numba_time.average * 1e6:.1f} us")
print(f"NumPy: {exercise_numpy_time.average * 1e6:.1f} us")

## Takeaway

- Time the current version so you know where the program is slow.
- Prefer clear NumPy when it already expresses the work well.
- Try Numba for slow numerical Python loops.
- Check the answer, call the compiled function once, and then time later calls.

Optional next step: [`1_numpy.ipynb`](1_numpy.ipynb).